# Diagnostyka UV dla rekonstrukcji Lyα one-loop

Notebook analizuje ukończony bundle `lcdm_2021_massless_production.npz`. Celem nie jest wprowadzanie nowego modelu, lecz ustalenie, dlaczego fit zależy od technicznego cutoffu $k_{\rm UV}$. Sprawdzamy surowe kanały, kasowanie $P_{22}+P_{13}$, skumulowane całki $I_0,I_2,I_4$ oraz kontrolne zastępowanie pojedynczych całek ich wartościami dla $k_{\rm UV}=10\,h/\mathrm{Mpc}$.

Kontrolne zastępowanie całek **nie jest alternatywnym modelem fizycznym** i nie służy do raportowania finalnego $\chi^2$. Jest wyłącznie testem lokalizującym problem numeryczny lub teoretyczny.

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'lyalpha_one_loop']
    for candidate in candidates:
        if (candidate / 'lyalpha_pt').is_dir() and (candidate / 'data').is_dir():
            return candidate.resolve()
    raise FileNotFoundError('Nie znaleziono katalogu projektu lyalpha_one_loop.')


PROJECT_ROOT = find_project_root()
THEORY_PATH = PROJECT_ROOT / 'results' / 'lcdm_2021_massless_production.npz'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'uv_diagnostics_production'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from lyalpha_pt.data import load_dr12
from lyalpha_pt.fit import EffectiveModelFit
from lyalpha_pt.theory import load_theory

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.precision', 6)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('THEORY_PATH  =', THEORY_PATH)
print('OUTPUT_DIR   =', OUTPUT_DIR)

In [ ]:
theory = load_theory(THEORY_PATH)
dataset = load_dr12(PROJECT_ROOT / 'data')

numerics = theory.metadata['numerics']
summary = pd.DataFrame({
    'quantity': [
        'model', 'redshifts', 'number of loop k points', 'loop k range [h/Mpc]',
        'k_trust [h/Mpc]', 'bundle digest', 'number of fitted data points'
    ],
    'value': [
        theory.metadata['model']['name'], ', '.join(f'{z:.1f}' for z in theory.z),
        len(theory.k_loop), f'{theory.k_loop[0]:.4g} -- {theory.k_loop[-1]:.4g}',
        numerics['k_trust'], theory.metadata['bundle_digest'], dataset.n_data
    ]
})
display(summary)

max_k_parallel = []
for iz, z in enumerate(theory.z):
    mask = np.isclose(dataset.z, z)
    max_k_parallel.append({
        'z': z,
        'max k_parallel [h/Mpc]': np.max(
            theory.velocity_to_hmpc[iz] * dataset.k_velocity[mask]
        )
    })
display(pd.DataFrame(max_k_parallel))

## 1. Surowe kanały one-loop

Pokazujemy stosunek pełnego kanału $P_{XY}^{\rm tree+1L}$ do tree level. Pionowe linie oznaczają przybliżony zakres danych oraz techniczne cutoffy 10 i $20\,h/\mathrm{Mpc}$.

In [ ]:
channel_labels = [r'$P_{\delta\delta}$', r'$P_{\delta\theta}$', r'$P_{\theta\theta}$']
colors = plt.cm.viridis(np.linspace(0.05, 0.95, len(theory.z)))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharex=True)
ratio_rows = []
for ich, (ax, label) in enumerate(zip(axes, channel_labels)):
    for iz, (z, color) in enumerate(zip(theory.z, colors)):
        ratio = theory.channels_one_loop[iz, :, ich] / theory.p_tree[iz]
        ax.plot(theory.k_loop, ratio, color=color, lw=1.8, label=f'z={z:.1f}')
        trusted = theory.k_loop <= 2.5
        ratio_rows.append({
            'z': z, 'channel': label,
            'min ratio all k': ratio.min(), 'max ratio all k': ratio.max(),
            'max ratio k<=2.5': ratio[trusted].max(),
            'negative points': int(np.sum(theory.channels_one_loop[iz, :, ich] < 0)),
        })
    ax.axvline(2.5, color='0.35', ls=':', lw=1.5)
    ax.axvline(10.0, color='tab:orange', ls='--', lw=1.2)
    ax.axvline(20.0, color='tab:red', ls='--', lw=1.2)
    ax.axhline(1.0, color='black', lw=1.0)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$k\,[h/\mathrm{Mpc}]$')
    ax.set_title(label)
axes[0].set_ylabel(r'$P_{XY}^{\rm tree+1L}/P_{\rm tree}$')
axes[-1].legend(ncol=1, fontsize=8)
fig.suptitle('Surowe kanały production: wzrost ogona UV')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'raw_channel_ratios_to_20.png', dpi=180, bbox_inches='tight')
plt.show()

ratio_table = pd.DataFrame(ratio_rows)
display(ratio_table)
ratio_table.to_csv(OUTPUT_DIR / 'raw_channel_ratio_summary.csv', index=False)

## 2. Kasowanie $P_{22}$ i $P_{13}$

Duże, przeciwnego znaku składniki mogą dawać umiarkowaną sumę. Pokazujemy je dla kanału gęstościowego.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 8), sharex=True, sharey=True)
axes = axes.ravel()
for iz, (z, ax) in enumerate(zip(theory.z, axes)):
    tree = theory.p_tree[iz]
    p22 = theory.loop_weight * theory.p22[iz, :, 0] / tree
    p13 = theory.loop_weight * theory.p13[iz, :, 0] / tree
    ax.plot(theory.k_loop, p22, lw=1.6, label=r'$P_{22}/P_{\rm tree}$')
    ax.plot(theory.k_loop, p13, lw=1.6, label=r'$P_{13}/P_{\rm tree}$')
    ax.plot(theory.k_loop, p22 + p13, color='black', lw=2.0, label=r'$(P_{22}+P_{13})/P_{\rm tree}$')
    ax.axhline(0, color='0.4', lw=0.8)
    ax.axvline(2.5, color='0.4', ls=':', lw=1.0)
    ax.axvline(10, color='tab:orange', ls='--', lw=1.0)
    ax.set_xscale('log')
    ax.set_yscale('symlog', linthresh=0.1)
    ax.set_title(f'z={z:.1f}')
    ax.set_xlabel(r'$k\,[h/\mathrm{Mpc}]$')
    ax.set_ylabel('relative correction')
axes[0].legend(fontsize=8)
axes[-1].axis('off')
fig.suptitle(r'Kasowanie $P_{22}$ i $P_{13}$ w kanale $\delta\delta$')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'p22_p13_cancellation_density.png', dpi=180, bbox_inches='tight')
plt.show()

## 3. Skumulowane $I_0$, $I_2$, $I_4$

Dla każdego cutoffu budujemy dokładnie te same całki, których używa fitter. Następnie mierzymy zmianę RMS względem $k_{\rm UV}=10\,h/\mathrm{Mpc}$.

In [ ]:
cutoffs = np.array([2.5, 3.0, 4.0, 5.0, 7.5, 10.0, 12.5, 15.0, 17.5, 20.0])
integrals = {}
for cutoff in cutoffs:
    fitter = EffectiveModelFit(
        dataset, theory, mode='one_loop', k_uv_cut=float(cutoff),
        covariance_mode='paper_diag'
    )
    integrals[cutoff] = np.column_stack([fitter.i0, fitter.i2, fitter.i4])

base = integrals[10.0]
sensitivity_rows = []
for cutoff in cutoffs:
    delta = integrals[cutoff] - base
    relative_rms = np.sqrt(np.mean(delta**2, axis=0)) / np.sqrt(np.mean(base**2, axis=0))
    sensitivity_rows.append({
        'k_uv': cutoff,
        'RMS change I0 / RMS I0(10)': relative_rms[0],
        'RMS change I2 / RMS I2(10)': relative_rms[1],
        'RMS change I4 / RMS I4(10)': relative_rms[2],
    })
sensitivity_table = pd.DataFrame(sensitivity_rows)
display(sensitivity_table)
sensitivity_table.to_csv(OUTPUT_DIR / 'integral_cutoff_sensitivity.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5))
for column, label in zip(sensitivity_table.columns[1:], [r'$I_0$', r'$I_2$', r'$I_4$']):
    ax.plot(sensitivity_table['k_uv'], sensitivity_table[column], 'o-', lw=2, label=label)
ax.axvline(10, color='0.4', ls='--')
ax.axvline(20, color='0.4', ls='--')
ax.set_yscale('log')
ax.set_xlabel(r'$k_{\rm UV}\,[h/\mathrm{Mpc}]$')
ax.set_ylabel('relative RMS change with respect to cutoff 10')
ax.set_title('Wrażliwość skumulowanych całek na cutoff')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'integral_cutoff_sensitivity.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
selected_z = [3.0, 4.2]
selected_kv = [0.005, 0.010, 0.020]
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
for row, z in enumerate(selected_z):
    zmask = np.isclose(dataset.z, z)
    global_indices = np.flatnonzero(zmask)
    for col, (component, label) in enumerate(zip(range(3), [r'$I_0$', r'$I_2$', r'$I_4$'])):
        ax = axes[row, col]
        for kv in selected_kv:
            local = np.argmin(np.abs(dataset.k_velocity[zmask] - kv))
            index = global_indices[local]
            values = np.array([integrals[c][index, component] for c in cutoffs])
            reference = integrals[10.0][index, component]
            ax.plot(cutoffs, values / reference, 'o-', label=fr'$k_v={dataset.k_velocity[index]:.3f}$')
        ax.axhline(1, color='black', lw=0.9)
        ax.axvline(10, color='0.4', ls='--', lw=1.0)
        ax.set_title(f'z={z:.1f}, {label}')
        ax.set_xlabel(r'$k_{\rm UV}\,[h/\mathrm{Mpc}]$')
        ax.set_ylabel('integral / value at cutoff 10')
        ax.legend(fontsize=8)
fig.suptitle('Skumulowane całki dla wybranych punktów danych')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'cumulative_integrals_selected_points.png', dpi=180, bbox_inches='tight')
plt.show()

## 4. Czy różnica $I_0(20)-I_0(10)$ ma postać countertermu?

Wkład z przedziału $10<k<20$ powinien być niezależny od dolnej granicy $k_\parallel$ i możliwie dobrze opisywany potęgą redshiftu.

In [ ]:
delta_i0_rows = []
for z in theory.z:
    mask = np.isclose(dataset.z, z)
    delta = integrals[20.0][mask, 0] - integrals[10.0][mask, 0]
    delta_i0_rows.append({
        'z': z, 'mean delta I0': np.mean(delta),
        'peak-to-peak across k_parallel': np.ptp(delta),
    })
delta_i0_table = pd.DataFrame(delta_i0_rows)
ratio_z = (1 + delta_i0_table['z'].to_numpy()) / 4.0
y = np.abs(delta_i0_table['mean delta I0'].to_numpy())
slope, intercept = np.polyfit(np.log(ratio_z), np.log(y), 1)
prediction = np.exp(intercept) * ratio_z**slope
delta_i0_table['power-law prediction'] = prediction
delta_i0_table['relative residual'] = y / prediction - 1
display(delta_i0_table)
print(f'Best power-law exponent: {slope:.6f}')
print(f'Maximum relative residual: {np.max(np.abs(delta_i0_table["relative residual"])):.6%}')
delta_i0_table.to_csv(OUTPUT_DIR / 'delta_i0_10_to_20_power_law.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(delta_i0_table['z'], y, 'o', ms=7, label='calculated')
ax.plot(delta_i0_table['z'], prediction, '-', lw=2, label=f'power law, exponent={slope:.3f}')
ax.set_xlabel('z')
ax.set_ylabel(r'$|I_0(20)-I_0(10)|$')
ax.set_title('Redshift dependence of the UV contribution to I0')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'delta_i0_redshift_power_law.png', dpi=180, bbox_inches='tight')
plt.show()

## 5. Wyniki fitu i bezpośredni test integralny

Pierwsza tabela zawiera rzeczywiste minima pełnego modelu. Druga tabela przedstawia kontrolny fit przy cutoffie 20, w którym wybrane całki zastępowano ich wartościami dla cutoffu 10. Wszystkie liczby pochodzą z wielostartowej optymalizacji sześciu parametrów.

In [ ]:
fit_cutoff_table = pd.DataFrame([
    {'k_uv': 10.0, 'chi2_one_loop': 198.386524615, 'note': 'counterterm bound removed; asymptotic minimum'},
    {'k_uv': 15.0, 'chi2_one_loop': 227.653872088, 'note': 'broad six-parameter fit'},
    {'k_uv': 20.0, 'chi2_one_loop': 233.415633516, 'note': 'expanded counterterm fit'},
])
fit_cutoff_table['delta_chi2_vs_linear_206.4704'] = fit_cutoff_table['chi2_one_loop'] - 206.470438314
display(fit_cutoff_table)
fit_cutoff_table.to_csv(OUTPUT_DIR / 'production_chi2_cutoff_summary.csv', index=False)

replacement_fit_table = pd.DataFrame([
    {'integrals used in cutoff-20 fitter': 'I0(20), I2(20), I4(20)', 'chi2': 233.415633516},
    {'integrals used in cutoff-20 fitter': 'I0(10), I2(20), I4(20)', 'chi2': 231.080920092},
    {'integrals used in cutoff-20 fitter': 'I0(20), I2(10), I4(20)', 'chi2': 217.463419659},
    {'integrals used in cutoff-20 fitter': 'I0(20), I2(20), I4(10)', 'chi2': 235.096617345},
    {'integrals used in cutoff-20 fitter': 'I0(10), I2(10), I4(20)', 'chi2': 197.497383617},
    {'integrals used in cutoff-20 fitter': 'I0(10), I2(20), I4(10)', 'chi2': 232.977830758},
    {'integrals used in cutoff-20 fitter': 'I0(20), I2(10), I4(10)', 'chi2': 216.844229287},
    {'integrals used in cutoff-20 fitter': 'I0(10), I2(10), I4(10)', 'chi2': 198.386528681},
])
display(replacement_fit_table)
replacement_fit_table.to_csv(OUTPUT_DIR / 'integral_replacement_fit_diagnostic.csv', index=False)

fig, ax = plt.subplots(figsize=(11, 4.8))
order = np.arange(len(replacement_fit_table))
ax.bar(order, replacement_fit_table['chi2'], color=['tab:red', 'tab:orange', 'tab:blue', 'tab:purple', 'tab:green', 'tab:gray', 'tab:cyan', 'tab:green'])
ax.axhline(206.470438314, color='black', ls='--', label='linear 206.4704')
ax.axhline(193.4, color='tab:green', ls=':', lw=2, label='paper one-loop 193.4')
ax.set_xticks(order, replacement_fit_table['integrals used in cutoff-20 fitter'], rotation=35, ha='right')
ax.set_ylabel(r'$\chi^2$')
ax.set_title('Która skumulowana całka odpowiada za niestabilność?')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'integral_replacement_chi2.png', dpi=180, bbox_inches='tight')
plt.show()

## 6. Test stałego przeskalowania poprawki one-loop

Używamy diagnostycznej rodziny $P_{XY}=P_{\rm tree}+\lambda(P_{XY}^{\rm 1L}-P_{\rm tree})$. Jeśli problemem był wyłącznie brakujący stały czynnik normalizacyjny, jedna wartość $\lambda$ powinna poprawiać oba cutoffy. Wyniki uzyskano przez ciągłe śledzenie tej samej gałęzi minimum.

In [ ]:
loop_scale_table = pd.DataFrame({
    'lambda': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'chi2_k10': [229.6587, 218.3116, 210.9529, 206.2030, 203.1524, 201.2080, 199.9837, 199.2276, 198.7751, 198.5183, 198.3865],
    'chi2_k20': [206.7606, 244.2149, 407.9273, 360.3634, 275.7614, 241.4894, 228.3956, 225.3410, 226.9759, 230.2637, 233.4156],
})
display(loop_scale_table)
loop_scale_table.to_csv(OUTPUT_DIR / 'loop_correction_scale_scan.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(loop_scale_table['lambda'], loop_scale_table['chi2_k10'], 'o-', lw=2, label=r'$k_{\rm UV}=10$')
ax.plot(loop_scale_table['lambda'], loop_scale_table['chi2_k20'], 's-', lw=2, label=r'$k_{\rm UV}=20$')
ax.axhline(206.470438314, color='black', ls='--', label='linear 206.4704')
ax.axhline(193.4, color='tab:green', ls=':', lw=2, label='paper 193.4')
ax.set_xlabel(r'loop correction scale $\lambda$')
ax.set_ylabel(r'$\chi^2$')
ax.set_title('Stałe przeskalowanie nie usuwa zależności od cutoffu')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'loop_correction_scale_scan.png', dpi=180, bbox_inches='tight')
plt.show()

## Wniosek

1. Bundle `production` nie wykazuje oscylacji interpolacji ani ujemnych kanałów, ale dla $k\sim20\,h/\mathrm{Mpc}$ pełne widma są około 20–40 razy większe od tree level.
2. Między cutoffem 10 i 20 zmiana RMS wynosi około 36% dla $I_0$, 2.6% dla $I_2$ i 0.13% dla $I_4$.
3. Dodatkowy wkład UV do $I_0$ jest praktycznie stały względem $k_\parallel$ i ma niemal idealną potęgową zależność od redshiftu, dlatego sam $I_0$ nie wyjaśnia zmiany $\chi^2$.
4. Bezpośrednie zastąpienie pokazuje, że krytyczna jest zmiana $I_2$, czyli kanału $P_{\delta\theta}$, przy współdziałaniu z przesunięciem $I_0$. $I_4/P_{\theta\theta}$ jest stabilne i nie powoduje problemu.
5. Stałe przeskalowanie całej poprawki nie naprawia jednocześnie cutoffów 10 i 20, więc problem nie wygląda na pojedynczy brakujący czynnik normalizacyjny.

**Następny test powinien porównać surowe $P_{\delta\theta}$ z niezależną implementacją one-loop, najpierw dla $z=3$ i $k\le2.5\,h/\mathrm{Mpc}$, a potem zbadać jego ogon do 20. Dopiero po tej kontroli warto ponownie uruchamiać pełne `precision`.**